# [사전작업] 챗봇 기본구조 확인 (Streamlit)

#### 필요한 라이브러리를 설치합니다. 일부 라이브러리와 호환 에러가 발생할 수 있지만, 실습과 관계 없으므로 계속 진행합니다.

In [1]:
!pip install -r requirements.txt -q

In [2]:
# 사전 정의된 기본 챗봇 애플리케이션 
!cp ./basic-chat.py ../demo-app.py

# [Lab1] 지식 문서 업로드 기능 구현 

In [4]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [5]:
import re
import json
import pdfplumber
from datetime import datetime
from langchain_core.documents import Document

In [6]:
import boto3
from utils.ssm import parameter_store

region=boto3.Session().region_name
pm = parameter_store(region)

domain_endpoint = pm.get_params(key="opensearch_domain_endpoint", enc=False)
opensearch_domain_endpoint = f"https://{domain_endpoint}"
opensearch_user_id = pm.get_params(key="opensearch_user_id", enc=False)
opensearch_user_password = pm.get_params(key="opensearch_user_password", enc=True)

In [7]:
with open('../libs/opensearch.yml', 'r') as file:
    file_contents = file.read()

modified_contents = file_contents.replace("{opensearch_domain_endpoint}", opensearch_domain_endpoint)

with open('../libs/opensearch.yml', 'w') as file:
    file.write(modified_contents)

opensearch_domain_endpoint

'https://search-rag-hol-gen-ai-workshop-ufxejvnxnnvkj2fyj2ze4mjrdi.us-west-2.es.amazonaws.com'

### PDF 문서 처리방식 확인

문서를 어떻게 chunking 할 것인지는 RAG 성능에 많은 영향을 미칩니다.
아래 예시에서는 Chunking 방식을 이해하기 위해, PDF 문서를 Page 단위에 맞춰 Low-level chunking 하도록 했습니다.

실제 활용에서는 Loader + Splitter 라이브러리를 사용해 Chunking 하는 경우가 많습니다.

- Loader 유형으로는 어떤 입력 문서들을 로드할 것인지 결정하며, PDF/CSV/JSON/Markdown/HTML/MS Office 등 다양한 포맷에 맞는 라이브러리를 지원합니다.
- Splitter 유형으로는 HTML 헤더/섹션, 특정 단어, 토큰 수 등 다양한 방법으로 결정합니다. 하나의 Chunk에 가능한 완전한 컨텍스트가 담길 수 있도록, 문서 패턴에 맞추는 것이 중요합니다.

In [8]:
pdf_list = ['./data/sample1_ko.pdf']

In [9]:
# PDF에서 cid를 추출해서 ASCII 문자로 변환
def text_pruner(text, current_pdf_file):
    def replace_cid(match):
        ascii_num = int(match.group(1))
        try:
            return chr(ascii_num)
        except:
            return ''  # 변환 실패 시 빈 문자열로 처리
    cid_pattern = re.compile(r'\(cid:(\d+)\)')
    return re.sub(cid_pattern, replace_cid, text)

# PDF 파일 처리 함수
def process_pdf(pdf_file):
    print(f"Processing PDF file: {pdf_file}")
    docs = []
    source_name = pdf_file.split('/')[-1]
    type_name = source_name.split(' ')[-1].replace('.pdf', '')

    with pdfplumber.open(pdf_file) as pdf:
        for page_number, page in enumerate(pdf.pages, start=1):
            page_text = page.extract_text()
            if page_text:
                pruned_text = text_pruner(page_text, pdf_file)
            else:
                pruned_text = ""
            # 텍스트 길이가 20 이상인 경우에만 Documnet로 저장
            if len(pruned_text) >= 20:  
                doc = Document(
                    page_content=pruned_text.replace('\n', ' '),
                    metadata={
                        "source": source_name,
                        "type": type_name,
                        "timestamp": datetime.now().isoformat()
                    }
                )
                docs.append(doc)
    if docs:
        load_document(docs)

def load_document(docs):
    # 동작방식을 확인하기 위해, 출력만 진행 (업데이트 예정)
    print("sample doc: ", docs[0])
    print("number of docs", len(docs))    

for pdf_file in pdf_list:
    process_pdf(pdf_file)


Processing PDF file: ./data/sample1_ko.pdf
sample doc:  page_content='해외여행보험 보통약관 제1조(보험계약의 성립) ①보험계약은 보험계약자의 청약과 보험회사의 승낙으로 이루어집니 다.(이하 "보험계약"은 "계약","보험계약자"는 "계약자","보험회사"는 "회사"라 합니다) ② 회사는 계약의 청약을 받고 보험료 전액 또는 제1회 보험료(일정기간 단위의 분할보험료) 를 받은 경우에는 청약일(진단계약의 경우에는 진단일)로부터 30일 이내에 승낙 또는 거절의 통지를 하며 통지가 없으면 승낙한 것으로 봅니다. ③ 회사가 청약을 승낙한 때에는 지체없이 보험증권을 계약자에게 교부하여 드리며, 청약을 거절할 경우에는 거절통지와 함께 받은 금액을 계약자에게 돌려드립니다. ④ 이미 성립한 계약을 연장하거나 변경하는 경우에는 회사는 보험증권에 그 사실을 기재하거 나 서면으로 알림으로써 보험증권의 교부에 대신할 수 있습니다. 제2조(약관교부 및 설명의무 등) ① 회사는 계약을 체결할 때 계약자에게 보험약관을 드리고 그 약관의 중요한 내용을 설명하여 드립니다. ② 회사가 제1항에 의해 제공될 약관을 계약자에게 전달하지 아니하거나 약관의 중요한 내용 을 설명하지 아니한 때 또는 계약체결시 계약자가 청약서에 자필서명(날인을 포함합니다)을 하지 아니한 때에는 계약자는 계약일로부터 1월 이내에 계약을 취소할 수 있습니다. ③ 제2항에 따라 계약이 취소된 경우에는 회사는 이미 납입한 보험료를 계약자에게 돌려 드리 며, 보험료를 받은 기간에 대하여 보험개발원이 공시하는 정기예금이율로 계산한 금액을 더 하여 지급합니다. 제3조(보험료) ① 보험료는 다른 약정이 없으면 보험기간이 시작되기 전에 내어야 합니다. ② 다른 약정이 없으면 보험기간이 시작된 후라도 보험료를 받기 전에 생긴 손해는 보상하여 드리지 아니합니다. 제4조(회사의 책임의 시기 및 종기) ① 회사의 책임은 보험기간의 첫날 오후 4시에 시작하여 마지막날 오후 4시에 끝납니다. 그

### OpenSearch를 지식 저장소로 활용

이제 지식 저장소로 OpenSearch를 활용하기 위해, 각 Document들을 보관합니다.

#### OpenSearch 클라이언트 생성

In [10]:
from opensearchpy import OpenSearch, RequestsHttpConnection

In [12]:
http_auth = (opensearch_user_id, opensearch_user_password)
os_client = OpenSearch(
            hosts=[
                {
                    'host': opensearch_domain_endpoint.replace("https://", ""),
                    'port': 443
                }
            ],
            http_auth=http_auth, 
            use_ssl=True,
            verify_certs=True,
            timeout=300,
            connection_class=RequestsHttpConnection
        )

#### OpenSearch에 `sample_index` 인덱스 생성

In [13]:
with open('index_template.json', 'r') as f:
    index_body = json.load(f)

index_name = "sample_index"
exists = os_client.indices.exists(index_name)

if exists:
    os_client.indices.delete(index=index_name)
    print("Existing index has been deleted. Create new one.")
else:
    print("Index does not exist, Create one.")

os_client.indices.create(index_name, body=index_body)

Index does not exist, Create one.


{'acknowledged': True, 'shards_acknowledged': True, 'index': 'sample_index'}

In [14]:
from botocore.config import Config

In [15]:
retry_config = Config(
        region_name=region,
        retries={
            "max_attempts": 10,
            "mode": "standard",
        },
    )
boto3_bedrock = boto3.client("bedrock-runtime", region_name=region, config=retry_config)

#### 벡터 임베딩 및 OpenSearch 벡터 저장/검색을 위한 클래스 생성

검색증강생성(RAG)의 자연어 기반 챗봇이 가능한 핵심 원리 중 하나는 벡터임베딩을 활용한 텍스트의 저장 및 검색입니다.

1. 자연어 텍스트를 벡터임베딩으로 변환해주는 Bedrock Embedding 모델을 정의하고,
2. OpenSearch에서 제공하는 벡터 저장/검색을 위한 클래스(`OpenSearchVectorSearch`)를 생성합니다.

In [16]:
from langchain.embeddings import BedrockEmbeddings
from langchain_community.vectorstores import OpenSearchVectorSearch

In [17]:
llmemb = BedrockEmbeddings(
    client=boto3_bedrock,
    model_id="amazon.titan-embed-g1-text-02"
)
dimension = 1536

vector_db = OpenSearchVectorSearch(
    index_name=index_name,
    opensearch_url=opensearch_domain_endpoint,
    embedding_function=llmemb,
    http_auth=http_auth,
)

#### 이제 문서 로드 함수 `load_document()`에 add_documents() 호출 구문을 추가하여, 실제 문서가 벡터화(vectorization)되어 OpenSearch에 추가되도록 합니다.

앞에서는 동작방식을 확인하기 위해, 문서를 print로만 출력하고 종료했었습니다.

In [18]:
def load_document(docs):
    vector_db.add_documents(docs)

for pdf_file in pdf_list:
    process_pdf(pdf_file)

Processing PDF file: ./data/sample1_ko.pdf


실행이 끝난 후에 OpenSearch에서 `sample_pdf` 인덱스를 조회해보면, 텍스트와 이 텍스트의 문맥적 의미를 담는 벡터가 함께 저장된 것이 확인됩니다.

(아래 내용은 참고만 하셔도 됩니다)

<img src="image/uploader-1.png">

# OpenSearch에서 인덱싱 결과 확인
#### ID raguser
#### PW MarsEarth1!


### 검색
```
GET sample_index/_search
{
  "query": {
    "match_all": {}
  },
  "size": 1
}
```

In [21]:
vector_db.similarity_search("특별비용담보 특별약관에서 회사가 보상하는 비용의 범위는?")

[Document(page_content='10만원을 한도로 합니다. 제3조(보상하지 아니하는 손해) 회사는 보통약관 제7조(보상하지 아니하는 손해) 제1항 제1호 내지 제3호, 제7호 내지 제12호의 사유로 인하여 생긴 손해는 보상하여 드리지 아니합니다. 제4조(보험금의 지급) 회사는 제2조(비용의 범위)의 비용중 정당하다고 인정된 부분에 대해서 만 보상하여 드리며, 계약자, 피보험자 또는 보험수익자가 타인으로부터 손해배상을 받을 수 있는 경우에는 그 금액을 지급하지 아니합니다. 제5조(보험금의 분담) 제1조(보상하는 손해)의 비용에 대하여 보험금을 지급할 다수의 계약이 체결되어 있는 경우에는 각각의 계약에 대하여 다른 계약이 없는 것으로 하여 산출한 보상책 임액의 합계액이 그 비용을 초과했을 때 회사는 이 계약에 따른 보상책임액의 위의 합계액에 대한 비율에 따라 보험금을 지급하여 드립니다. 제6조(보상한도액) 회사가 이 특별약관에 관하여 지급할 보험금은 보험기간을 통하여 이 특별 약관의 보험가입금액을 한도로 합니다. 제7조(준용규정) 이 특별약관에 정하지 아니한 사항은 보통약관을 따릅니다.', metadata={'source': 'sample1_ko.pdf', 'type': 'sample1_ko', 'timestamp': '2024-12-07T06:38:27.544263'}),
 Document(page_content='특별비용담보 특별약관 제1조(보상하는 손해) ① 회사는 아래의 사유로 계약자, 피보험자 또는 피보험자의 법정상속 인이 부담하는 비용을 이 특별약관에 따라 보상하여 드립니다. 1. 보통약관 제6조(보상하는 손해)의 여행도중(이하「여행도중」이라 합니다)에 피보험자 가 탑승한 항공기 또는 선박이 행방불명 또는 조난된 경우 또는 산악등반 중에 조난된 경 우 2. 여행도중에 급격하고도 우연한 외래의 사고에 따라 긴급수색구조등이 필요한 상태로 된 것이 경찰등의 공공기관에 의하여 확인된 경우 3. 보통약관 제6조(보상하는 손해)의 상해를 직접원인으로 하

### 정상적으로 Document가 조회된다면, 이제 문서를 검색할 수 있습니다.

In [ ]:
# index_name = "rag_index"
# pdf_list = ['./data/sample2_ko.pdf']

# def load_document(docs):
#     vector_db.add_documents(docs)

# for pdf_file in pdf_list:
#     process_pdf(pdf_file)